In [1]:
pip install ultralytics opencv-python numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install pygame

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install --upgrade pygame setuptools

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install pywhatkit ultralytics opencv-python

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install twilio

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import platform
import threading
import time
import cv2
import numpy as np
import pywhatkit as pwk
from twilio.rest import Client
from ultralytics import YOLO

if platform.system() == "Windows":
  import ctypes
  import winsound

# --- Twilio Credentials ---
TWILIO_ACCOUNT_SID = "your_account_sid_here"
TWILIO_AUTH_TOKEN = "your_auth_token_here"
TWILIO_PHONE_NUMBER = "+919309097951"


class SmartSurveillanceSystem:

  def __init__(
      self,
      model_path="yolov8n.pt",
      source=0,
      conf_threshold=0.45,
      alert_fps=60,
      target_phone_number="+919309097951",
  ):
    print(f"[INFO] Loading YOLO model: {model_path}...")
    self.model = YOLO(model_path)
    self.source = source
    self.conf_threshold = conf_threshold
    self.alert_fps = alert_fps
    self.last_alert_time = 0
    self.is_playing_siren = False
    self.is_sending_whatsapp = False
    self.target_phone_number = target_phone_number

    self.message_seen = False
    self.awaiting_seen = False

    self.surveillance_classes = [
        "person",
        "car",
        "motorcycle",
        "bus",
        "truck",
        "backpack",
        "handbag",
        "suitcase",
    ]

  def maximize_system_volume(self):
    """Windows वर सिस्टीमचा आवाज ऑटोमॅटिक १००% वर वाढवतो."""
    if platform.system() == "Windows":
      try:
        # Volume Up Key (50 वेळा दाबून full volume करणे)
        for _ in range(50):
          ctypes.windll.user32.keybd_event(0xAF, 0, 0, 0)
          ctypes.windll.user32.keybd_event(0xAF, 0, 2, 0)
      except Exception as e:
        print(f"[WARNING] Could not set volume: {e}")

  def play_continuous_siren(self):
    """जोपर्यंत मेसेज सीन (Acknowledged) होत नाही तोपर्यंत सतत मोठा आवाज करेल."""
    self.is_playing_siren = True
    self.maximize_system_volume()  # आवाज कमाल (Max) करणे

    system_os = platform.system()
    print("🚨 [SIREN STARTED] Playing loud continuous alert...")

    while self.is_playing_siren and self.awaiting_seen:
      try:
        if system_os == "Windows":
          # उच्च फ्रिक्वेन्सी (2500Hz) वर मोठा Beep आवाज
          winsound.Beep(2500, 1000)
        elif system_os == "Darwin":
          os.system(
              'say -v Alert "INTRUSION DETECTED! CHECK YOUR PHONE IMMEDIATELY"'
          )
        else:
          os.system('spd-say "INTRUSION DETECTED! CHECK YOUR PHONE"')
        time.sleep(0.1)
      except Exception as e:
        print(f"[WARNING] Siren Error: {e}")
        break

    print("🛑 [SIREN STOPPED] Siren alert stopped.")
    self.is_playing_siren = False

  def make_emergency_call(self):
    """Triggers emergency phone call via Twilio API."""
    try:
      print(
          f"📞 [EMERGENCY CALL] Calling {self.target_phone_number} via Twilio..."
      )
      client = Client(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN)

      call = client.calls.create(
          twiml="<Response><Say voice='alice'>Emergency Security Alert! Intrusion detected at your property. Please check immediately.</Say></Response>",
          to=self.target_phone_number,
          from_=TWILIO_PHONE_NUMBER,
      )
      print(f"✅ [CALL SUCCESS] Call SID: {call.sid}")
    except Exception as e:
      print(
          f"[ERROR] Failed to make emergency call. Check Twilio settings: {e}"
      )

  def monitor_message_timeout(self, timeout_seconds=60):
    """Monitors if message is acknowledged within 60 seconds."""
    self.awaiting_seen = True
    self.message_seen = False

    # १. सतत चालणारा मोठा सायरेन सुरू करणे
    if not self.is_playing_siren:
      siren_thread = threading.Thread(
          target=self.play_continuous_siren, daemon=True
      )
      siren_thread.start()

    print(
        f"⏳ [TIMER STARTED] Waiting {timeout_seconds}s (Press 's' on window to stop siren)..."
    )

    start_time = time.time()
    while time.time() - start_time < timeout_seconds:
      if self.message_seen:
        print("✅ [CONFIRMED] Message acknowledged! Stopping siren & call.")
        self.is_playing_siren = False
        self.awaiting_seen = False
        return
      time.sleep(0.5)

    # ६० सेकंदात प्रतिसाद न मिळाल्यास Call लावणे
    print(
        "⚠️ [NO RESPONSE] Message not acknowledged in 1 min! Making Emergency"
        " Call..."
    )
    self.is_playing_siren = False  # कॉल करण्यापूर्वी सायरेन थांबवणे
    self.make_emergency_call()
    self.awaiting_seen = False

  def send_whatsapp_text(self, message_text):
    """Sends automatic text via WhatsApp Web and starts timer."""
    self.is_sending_whatsapp = True
    try:
      print(
          f"[INFO] Sending WhatsApp text alert to {self.target_phone_number}..."
      )
      pwk.sendwhatmsg_instantly(
          phone_no=self.target_phone_number,
          message=message_text,
          wait_time=8,
          tab_close=True,
          close_time=2,
      )
      print("✅ [SUCCESS] WhatsApp Text Alert Sent!")

      # ६० सेकंदांचा टायमर आणि सायरेन थ्रेड सुरू करणे
      timer_thread = threading.Thread(
          target=self.monitor_message_timeout, args=(60,), daemon=True
      )
      timer_thread.start()

    except Exception as e:
      print(f"[ERROR] Failed to send WhatsApp alert: {e}")

    self.is_sending_whatsapp = False

  def trigger_alert(self, person, bag):
    """Action trigger for security alerts."""
    alert_msgs = []
    if person:
      alert_msgs.append("Human Intrusion Detected")
    if bag:
      alert_msgs.append("Unattended Baggage Detected")

    alert_summary = " | ".join(alert_msgs)
    timestamp = time.ctime()

    print(f"🚨 [SECURITY ALERT] {timestamp}: {alert_summary}")

    if not self.is_sending_whatsapp and not self.awaiting_seen:
      whatsapp_message = (
          f"🚨 *SECURITY ALERT DETECTED*\n\n"
          f"*Time:* {timestamp}\n"
          f"*Event:* {alert_summary}\n\n"
          f"_Reply or press 's' in system within 1 minute to stop Siren & Call._"
      )
      wa_thread = threading.Thread(
          target=self.send_whatsapp_text,
          args=(whatsapp_message,),
          daemon=True,
      )
      wa_thread.start()

  def process_stream(self):
    """Starts video capture and performs real-time detection."""
    cap = cv2.VideoCapture(self.source)

    if not cap.isOpened():
      print(f"[ERROR] Could not open video source: {self.source}")
      return

    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    print(
        "[INFO] Starting feed. Controls: Press 's' = Stop Siren / Mark Seen |"
        " Press 'q' = Exit."
    )

    prev_frame_time = time.time()

    while True:
      ret, frame = cap.read()
      if not ret:
        print("[WARNING] Frame capture failed or stream ended.")
        break

      results = self.model(
          frame, conf=self.conf_threshold, stream=True, verbose=False
      )

      person_detected = False
      unattended_bag = False

      for result in results:
        boxes = result.boxes
        for box in boxes:
          cls_id = int(box.cls[0])
          class_name = self.model.names[cls_id]
          conf = float(box.conf[0])

          if class_name in self.surveillance_classes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            if class_name == "person":
              color = (0, 0, 255)
              person_detected = True
            elif class_name in ["car", "motorcycle", "bus", "truck"]:
              color = (0, 255, 255)
            else:
              color = (255, 255, 0)
              unattended_bag = True

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            label = f"{class_name.capitalize()} {conf:.2f}"
            cv2.putText(
                frame,
                label,
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2,
            )

      current_time = time.time()
      if (
          person_detected or unattended_bag
      ) and current_time - self.last_alert_time > self.alert_fps:
        self.trigger_alert(person_detected, unattended_bag)
        self.last_alert_time = current_time

      # Display System Status on Screen
      status_text = (
          "🚨 SIREN RINGING (AWAITING SEEN)"
          if self.awaiting_seen
          else "MONITORING ACTIVE"
      )
      cv2.putText(
          frame,
          f"STATUS: {status_text}",
          (20, 40),
          cv2.FONT_HERSHEY_SIMPLEX,
          0.7,
          (0, 0, 255) if self.awaiting_seen else (0, 255, 0),
          2,
      )

      cv2.imshow("Smart Surveillance System (YOLOv8)", frame)

      key = cv2.waitKey(1) & 0xFF
      if key == ord("s"):  # Stop Siren & Cancel Call
        self.message_seen = True
        self.is_playing_siren = False
      elif key == ord("q"):
        self.is_playing_siren = False
        break

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
  TARGET_NUMBER = "+919309097951"

  surveillance = SmartSurveillanceSystem(
      model_path="yolov8n.pt",
      source=0,
      conf_threshold=0.5,
      alert_fps=60,
      target_phone_number=TARGET_NUMBER,
  )
  surveillance.process_stream()

[INFO] Loading YOLO model: yolov8n.pt...
[INFO] Starting feed. Controls: Press 's' = Stop Siren / Mark Seen | Press 'q' = Exit.
🚨 [SECURITY ALERT] Mon Oct 19 07:04:58 2026: Human Intrusion Detected
[INFO] Sending WhatsApp text alert to +919309097951...
✅ [SUCCESS] WhatsApp Text Alert Sent!
⏳ [TIMER STARTED] Waiting 60s (Press 's' on window to stop siren)...
🚨 [SIREN STARTED] Playing loud continuous alert...
